# Book Rating Prediction

This notebook builds a machine learning pipeline to predict a book's average rating from Goodreads-style metadata. We load and clean the data, engineer features, train and compare four regression models, look at feature importance, and save the final model for use in a Streamlit app.

## Loading the Data

The raw CSV has 11,128 lines. Some rows don't match the expected number of columns (usually caused by commas or quotes inside book titles), so we use `on_bad_lines='skip'` to skip them while loading. This leaves about 11,123 rows.

In [1]:
import pandas as pd

# on_bad_lines='skip' = don't crash on rows with the wrong number of fields
df = pd.read_csv("books.csv", on_bad_lines="skip")

print("Shape (rows, cols):", df.shape)
df.head()

Shape (rows, cols): (11123, 12)


,bookID,title,authors,average_rating,isbn,isbn13,language_code,num_pages,ratings_count,text_reviews_count,publication_date,publisher
0,1,Harry Potter and the Half-Blood Prince (Harry ...,J.K. Rowling/Mary GrandPré,4.57,0439785960,9780439785969,eng,652,2095690,27591,9/16/2006,Scholastic Inc.
1,2,Harry Potter and the Order of the Phoenix (Har...,J.K. Rowling/Mary GrandPré,4.49,0439358078,9780439358071,eng,870,2153167,29221,9/1/2004,Scholastic Inc.
2,4,Harry Potter and the Chamber of Secrets (Harry...,J.K. Rowling,4.42,0439554896,9780439554893,eng,352,6333,244,11/1/2003,Scholastic
3,5,Harry Potter and the Prisoner of Azkaban (Harr...,J.K. Rowling/Mary GrandPré,4.56,043965548X,9780439655484,eng,435,2339585,36325,5/1/2004,Scholastic Inc.
4,8,Harry Potter Boxed Set Books 1-5 (Harry Potte...,J.K. Rowling/Mary GrandPré,4.78,0439682584,9780439682589,eng,2690,41428,164,9/13/2004,Scholastic


We used `.info()`, `.describe()` and null counts to check the data types, value ranges and missing values before doing any cleaning.

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 11123 entries, 0 to 11122
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   bookID              11123 non-null  int64  
 1   title               11123 non-null  str    
 2   authors             11123 non-null  str    
 3   average_rating      11123 non-null  float64
 4   isbn                11123 non-null  str    
 5   isbn13              11123 non-null  int64  
 6   language_code       11123 non-null  str    
 7     num_pages         11123 non-null  int64  
 8   ratings_count       11123 non-null  int64  
 9   text_reviews_count  11123 non-null  int64  
 10  publication_date    11123 non-null  str    
 11  publisher           11123 non-null  str    
dtypes: float64(1), int64(5), str(6)
memory usage: 2.1 MB


In [3]:
df.describe()

,bookID,average_rating,isbn13,num_pages,ratings_count,text_reviews_count
count,11123.000000,11123.000000,1.112300e+04,11123.000000,1.112300e+04,11123.000000
mean,21310.856963,3.934075,9.759880e+12,336.405556,1.794285e+04,542.048099
std,13094.727252,0.350485,4.429758e+11,241.152626,1.124992e+05,2576.619589
min,1.000000,0.000000,8.987060e+09,0.000000,0.000000e+00,0.000000
25%,10277.500000,3.770000,9.780345e+12,192.000000,1.040000e+02,9.000000
50%,20287.000000,3.960000,9.780582e+12,299.000000,7.450000e+02,47.000000
75%,32104.500000,4.140000,9.780872e+12,416.000000,5.000500e+03,238.000000
max,45641.000000,5.000000,9.790008e+12,6576.000000,4.597666e+06,94265.000000


In [4]:
print("Null counts:")
print(df.isnull().sum())

print("\nColumn names (repr shows hidden spaces):")
for col in df.columns:
    print(repr(col))

Null counts:
bookID                0
title                 0
authors               0
average_rating        0
isbn                  0
isbn13                0
language_code         0
  num_pages           0
ratings_count         0
text_reviews_count    0
publication_date      0
publisher             0
dtype: int64

Column names (repr shows hidden spaces):
'bookID'
'title'
'authors'
'average_rating'
'isbn'
'isbn13'
'language_code'
'  num_pages'
'ratings_count'
'text_reviews_count'
'publication_date'
'publisher'


### Initial observations

- The dataset has 11,123 rows and 12 columns after loading.
- There are no NaN values, but missing data is represented as 0 instead. For example, `average_rating` and `num_pages` both have a minimum of 0.
- The `num_pages` column name has a leading space, which needs to be fixed before it can be used.
- `publication_date` is still stored as text, not as a real date.
- `bookID`, `isbn` and `isbn13` are identifiers and won't be used as features.
- The target column is `average_rating`.

## Data Cleaning

Four things need to be fixed before the data is ready for modelling: the `num_pages` column name has extra spaces, some `publication_date` values are invalid, `average_rating == 0` actually means the book has no rating, and `num_pages == 0` means the page count is missing.

### Fixing Column Names

The `num_pages` column has leading spaces in its name, so we strip whitespace from all column names.

In [5]:
df.columns = df.columns.str.strip()
print([repr(c) for c in df.columns])

["'bookID'", "'title'", "'authors'", "'average_rating'", "'isbn'", "'isbn13'", "'language_code'", "'num_pages'", "'ratings_count'", "'text_reviews_count'", "'publication_date'", "'publisher'"]


### Invalid Publication Dates

A few rows have impossible dates, such as 11/31/2000 (November only has 30 days). We parse the dates with `errors='coerce'`, which turns invalid dates into NaT, and drop those rows. Only 2 rows are affected.

In [6]:
df["publication_date_parsed"] = pd.to_datetime(
    df["publication_date"], format="%m/%d/%Y", errors="coerce"
)

bad_dates = df["publication_date_parsed"].isna()
print("Invalid publication_date rows:", bad_dates.sum())
print(df.loc[bad_dates, ["title", "publication_date"]].head())

df = df.loc[~bad_dates].copy()
print("Shape after dropping bad dates:", df.shape)

Invalid publication_date rows: 2
                                                   title publication_date
8177   In Pursuit of the Proper Sinner (Inspector Lyn...       11/31/2000
11094         Montaillou  village occitan de 1294 à 1324        6/31/1982
Shape after dropping bad dates: (11121, 13)


### Dropping Rows with average_rating == 0

A rating of exactly 0 doesn't really happen once a book has actual ratings, so we treat it as a missing label rather than a real score and drop these rows. 25 rows are affected.

In [7]:
print("Rows with average_rating == 0:", (df["average_rating"] == 0).sum())
df = df.loc[df["average_rating"] != 0].copy()
print("Shape after dropping zero ratings:", df.shape)

Rows with average_rating == 0: 25
Shape after dropping zero ratings: (11096, 13)


### Handling Missing Page Counts

About 76 books have `num_pages == 0`. Instead of dropping these rows, since they still have a valid rating and other useful features, we impute the missing page count with the median and add a `pages_missing_flag` column so the model can tell which rows were imputed.

In [8]:
df["pages_missing_flag"] = (df["num_pages"] == 0).astype(int)

median_pages = df.loc[df["num_pages"] > 0, "num_pages"].median()
print("Median pages (nonzero only):", median_pages)
print("Rows imputed:", df["pages_missing_flag"].sum())

df.loc[df["num_pages"] == 0, "num_pages"] = median_pages

print("num_pages == 0 remaining:", (df["num_pages"] == 0).sum())
print("Final shape after Step 2:", df.shape)
df[["num_pages", "pages_missing_flag", "average_rating"]].head()

Median pages (nonzero only): 302.0
Rows imputed: 76
num_pages == 0 remaining: 0
Final shape after Step 2: (11096, 14)


,num_pages,pages_missing_flag,average_rating
0,652,0,4.57
1,870,0,4.49
2,352,0,4.42
3,435,0,4.56
4,2690,0,4.78


### Summary of Cleaning Steps

| Action | Rule applied |
|---|---|
| Column names | stripped whitespace |
| Invalid dates | parsed, then dropped (~2 rows) |
| average_rating == 0 | dropped (~25 rows) |
| num_pages == 0 | median imputed + flagged (~76 rows) |

We treat these two zero-cases differently because `average_rating` is the target — we can't invent the label — while `num_pages` is just one of several features, so imputing it is reasonable.

## Feature Engineering

The raw columns aren't directly usable by the models, since part of the useful information is stored as text (`authors`, `publication_date`, `title`). This step creates six new features and keeps three of the original numeric columns, giving a 9-column feature matrix.

### publication_year

We only keep the year from the publication date, since that's enough to capture whether a book is older or more recent.

In [9]:
df["publication_year"] = df["publication_date_parsed"].dt.year
print(df["publication_year"].describe())
df[["publication_date", "publication_year"]].head()

count    11096.000000
mean      2000.176370
std          8.247074
min       1900.000000
25%       1998.000000
50%       2003.000000
75%       2005.000000
max       2020.000000
Name: publication_year, dtype: float64


,publication_date,publication_year
0,9/16/2006,2006
1,9/1/2004,2004
2,11/1/2003,2003
3,5/1/2004,2004
4,9/13/2004,2004


### author_count

The `authors` field lists multiple authors separated by `/` (co-authors or illustrators, for example). We count them to get a simple numeric feature for how many people contributed to the book.

In [10]:
df["author_count"] = df["authors"].str.split("/").str.len()
print(df["author_count"].value_counts().head(10))
df[["authors", "author_count"]].head()

author_count
1     6547
2     3069
3     1003
4      221
5       65
6       54
7       21
10      12
8       11
15      11
Name: count, dtype: int64


,authors,author_count
0,J.K. Rowling/Mary GrandPré,2
1,J.K. Rowling/Mary GrandPré,2
2,J.K. Rowling,1
3,J.K. Rowling/Mary GrandPré,2
4,J.K. Rowling/Mary GrandPré,2


### title_length

We used the character length of the title as a feature, rather than word count, since it's simpler to compute and doesn't depend on how the title happens to be split into words.

In [11]:
df["title_length"] = df["title"].str.len()  # character count
# Alternative: df["title_length"] = df["title"].str.split().str.len()  # word count
print(df["title_length"].describe())
df[["title", "title_length"]].head()

count    11096.000000
mean        35.691330
std         23.526032
min          2.000000
25%         18.000000
50%         31.000000
75%         47.000000
max        254.000000
Name: title_length, dtype: float64


,title,title_length
0,Harry Potter and the Half-Blood Prince (Harry ...,57
1,Harry Potter and the Order of the Phoenix (Har...,60
2,Harry Potter and the Chamber of Secrets (Harry...,58
3,Harry Potter and the Prisoner of Azkaban (Harr...,59
4,Harry Potter Boxed Set Books 1-5 (Harry Potte...,54


### language_code_encoded and publisher_encoded

Both `language_code` and `publisher` are categorical text columns, so we used `LabelEncoder` to convert them into integers. One-hot encoding wasn't really practical for `publisher`, since it has around 2,200 unique values, which would create way too many columns. Label encoding is fine here since the models we use are tree-based, and trees don't assume any ordering between the encoded values.

In [12]:
from sklearn.preprocessing import LabelEncoder

print("Unique languages:", df["language_code"].nunique())
print("Unique publishers:", df["publisher"].nunique())

language_encoder = LabelEncoder()
publisher_encoder = LabelEncoder()

df["language_code_encoded"] = language_encoder.fit_transform(df["language_code"])
df["publisher_encoded"] = publisher_encoder.fit_transform(df["publisher"])

df[["language_code", "language_code_encoded", "publisher", "publisher_encoded"]].head()

Unique languages: 26
Unique publishers: 2274


,language_code,language_code_encoded,publisher,publisher_encoded
0,eng,5,Scholastic Inc.,1762
1,eng,5,Scholastic Inc.,1762
2,eng,5,Scholastic,1757
3,eng,5,Scholastic Inc.,1762
4,eng,5,Scholastic,1757


### Final Feature Matrix

The final feature set has 9 columns: `num_pages`, `ratings_count`, `text_reviews_count`, `author_count`, `title_length`, `publication_year`, `language_code_encoded`, `publisher_encoded` and `pages_missing_flag`. The target is `average_rating`.

We excluded `bookID`, `isbn` and `isbn13`, since they just identify each book and don't carry any predictive information.

In [13]:
FEATURE_COLS = [
    "num_pages",
    "ratings_count",
    "text_reviews_count",
    "author_count",
    "title_length",
    "publication_year",
    "language_code_encoded",
    "publisher_encoded",
    "pages_missing_flag",
]

X = df[FEATURE_COLS].copy()
y = df["average_rating"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nFeature dtypes:")
print(X.dtypes)
X.head()

X shape: (11096, 9)
y shape: (11096,)

Feature dtypes:
num_pages                int64
ratings_count            int64
text_reviews_count       int64
author_count             int64
title_length             int64
publication_year         int32
language_code_encoded    int64
publisher_encoded        int64
pages_missing_flag       int64
dtype: object


,num_pages,ratings_count,text_reviews_count,author_count,title_length,publication_year,language_code_encoded,publisher_encoded,pages_missing_flag
0,652,2095690,27591,2,57,2006,5,1762,0
1,870,2153167,29221,2,60,2004,5,1762,0
2,352,6333,244,1,58,2003,5,1757,0
3,435,2339585,36325,2,59,2004,5,1762,0
4,2690,41428,164,2,54,2004,5,1757,0


### Note on Label Encoding

Label encoding works for tree-based models but wouldn't be appropriate for Linear Regression, since it would treat the encoded publisher numbers as if they had a real order (publisher 1500 isn't "more" than publisher 20). For a linear model, one-hot or target encoding would be better choices here.

## Train/Test Split

We split the data into training and test sets so we can evaluate the models on data they haven't seen before.

The data was split 80% training / 20% testing, using `random_state=42` so the split is reproducible. Since this is a regression problem (the target is continuous, not categorical), we didn't use stratified sampling.

In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train:", X_train.shape, y_train.shape)
print("Test: ", X_test.shape, y_test.shape)
print("\nTarget mean — train:", round(y_train.mean(), 4), "| test:", round(y_test.mean(), 4))
print("Target std  — train:", round(y_train.std(), 4), "| test:", round(y_test.std(), 4))

Train: (8876, 9) (8876,)
Test:  (2220, 9) (2220,)

Target mean — train: 3.94 | test: 3.9547
Target std  — train: 0.2982 | test: 0.2918


### Why We Fixed the Random Seed

Fixing `random_state=42` means the same split, and the same random components in Random Forest and Gradient Boosting, are used every time the notebook runs. This matters because some of our models ended up very close in performance (a MAE difference of about 0.0002 in one case), so without a fixed seed the comparison between models wouldn't be reliable.

## Model Training

We trained four regression models on the same training set and evaluated them on the same test set using MAE, RMSE and R².

### Linear Regression

Fits a straight-line relationship between the features and the target. Simple and fast, but can't capture non-linear relationships or interactions between features.

### Decision Tree Regressor

Splits the data into regions based on feature values. Can capture non-linear patterns, but a single tree tends to overfit the training data.

### Random Forest Regressor

Trains many decision trees on random subsets of the data and averages their predictions, which reduces overfitting compared to a single tree.

### Gradient Boosting Regressor

Builds trees one at a time, where each new tree tries to correct the errors of the previous ones. Usually performs well on tabular data like this.

In [15]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(
        n_estimators=100, random_state=42, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
}

trained = {}
predictions = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    trained[name] = model
    predictions[name] = model.predict(X_test)
    print(f"Fitted: {name}")

print("\nAll four models trained. Pause before the comparison table ↓")

Fitted: Linear Regression
Fitted: Decision Tree


Fitted: Random Forest


Fitted: Gradient Boosting

All four models trained. Pause before the comparison table ↓


### What We Expected

We expected the two ensemble methods (Random Forest and Gradient Boosting) to outperform a single Decision Tree and Linear Regression, since book ratings likely don't follow a simple linear relationship with the features, and a single tree is prone to overfitting.

In [16]:
rows = []
for name, pred in predictions.items():
    rows.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": mean_squared_error(y_test, pred) ** 0.5,
        "R²": r2_score(y_test, pred),
    })

results = pd.DataFrame(rows).sort_values("R²", ascending=False).reset_index(drop=True)
results_display = results.copy()
results_display[["MAE", "RMSE", "R²"]] = results_display[["MAE", "RMSE", "R²"]].round(4)
results_display

,Model,MAE,RMSE,R²
0,Gradient Boosting,0.1983,0.2689,0.1504
1,Random Forest,0.1981,0.2705,0.1398
2,Linear Regression,0.2122,0.2831,0.0580
3,Decision Tree,0.2819,0.3790,-0.6878


## Model Evaluation

Results on the test set:

| Model | MAE | RMSE | R² |
|---|---:|---:|---:|
| Gradient Boosting | 0.1983 | 0.2689 | 0.1504 |
| Random Forest | 0.1981 | 0.2705 | 0.1398 |
| Linear Regression | 0.2122 | 0.2831 | 0.0580 |
| Decision Tree | 0.2819 | 0.3790 | -0.6878 |

This matches what we expected. The Decision Tree's negative R² means it actually performs worse than just predicting the average rating every time, which points to overfitting. Random Forest and Gradient Boosting are close on MAE, but Gradient Boosting has better RMSE and R², so we picked it as the final model.

### Why R² Is Low

An R² around 0.15 isn't a sign that the model failed. Book ratings depend heavily on the actual content and quality of the book, which isn't captured by any of our features. We're only using metadata (pages, review counts, publisher, language, title length), and that correlates weakly with quality at best. A much higher R² here would actually be more suspicious, since it could point to data leakage.

### Feature Importance

We looked at feature importance from the Gradient Boosting model, since it was the best-performing one. This shows which features the model relied on most to reduce error — it doesn't mean those features cause a higher rating.

In [17]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

best_name = results.iloc[0]["Model"]
best_model = trained[best_name]
print("Interpreting:", best_name)

importances = pd.Series(
    best_model.feature_importances_, index=FEATURE_COLS
).sort_values(ascending=True)

print(importances.sort_values(ascending=False).round(4))

ax = importances.plot(kind="barh", figsize=(8, 4))
ax.set_xlabel("Importance")
ax.set_title(f"Feature importances — {best_name}")
plt.tight_layout()
plt.show()

Interpreting: Gradient Boosting
num_pages                0.2524
ratings_count            0.1878
title_length             0.1776
publisher_encoded        0.1615
publication_year         0.0644
author_count             0.0582
text_reviews_count       0.0549
language_code_encoded    0.0364
pages_missing_flag       0.0068
dtype: float64


/var/folders/n1/80d_f9r52nq0y103lc7ss9dc0000gn/T/ipykernel_20156/2936383914.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Results

The most important features were `num_pages` (0.25), `ratings_count` (0.19), `title_length` (0.18) and `publisher_encoded` (0.16). The rest — `publication_year`, `author_count`, `text_reviews_count`, `language_code_encoded` and `pages_missing_flag` — contributed much less.

`ratings_count` is a good example of a feature that shouldn't be read as causal: having more ratings doesn't cause a higher score. It's more likely that already well-liked books get rated more often, and that books with very few ratings tend to have more extreme, less reliable average scores.

## Model Saving

To use the model later in the app, we need more than just the trained model itself. The app receives raw inputs like `language_code="eng"`, so we also saved the label encoders, the feature column order, and the median value used for page imputation. Everything is bundled together with `joblib`.

In [18]:
import joblib
from pathlib import Path

models_dir = Path("models")
models_dir.mkdir(exist_ok=True)

# median_pages was computed in Step 2 — recreate from current df for the bundle
median_pages_value = float(df.loc[df["pages_missing_flag"] == 0, "num_pages"].median())

artifact = {
    "model": trained[best_name],
    "language_encoder": language_encoder,
    "publisher_encoder": publisher_encoder,
    "feature_cols": FEATURE_COLS,
    "median_pages": median_pages_value,
    "model_name": best_name,
    # Fallbacks for unseen categories at inference (must exist in encoder.classes_)
    "fallback_language": df["language_code"].mode().iloc[0],
    "fallback_publisher": df["publisher"].mode().iloc[0],
}

artifact_path = models_dir / "book_rating_bundle.joblib"
joblib.dump(artifact, artifact_path)

print(f"Saved bundle → {artifact_path}")
print("Keys:", list(artifact.keys()))
print("Feature order:", artifact["feature_cols"])
print("Median pages stored:", artifact["median_pages"])
print("Fallbacks:", artifact["fallback_language"], "|", artifact["fallback_publisher"])

Saved bundle → models/book_rating_bundle.joblib
Keys: ['model', 'language_encoder', 'publisher_encoder', 'feature_cols', 'median_pages', 'model_name', 'fallback_language', 'fallback_publisher']
Feature order: ['num_pages', 'ratings_count', 'text_reviews_count', 'author_count', 'title_length', 'publication_year', 'language_code_encoded', 'publisher_encoded', 'pages_missing_flag']
Median pages stored: 302.0
Fallbacks: eng | Vintage


In [19]:
# Sanity check: reload and score once — should match the table for that model
loaded = joblib.load(artifact_path)
reloaded_pred = loaded["model"].predict(X_test[loaded["feature_cols"]])
print("Reloaded R²:", round(r2_score(y_test, reloaded_pred), 4))
print("Matches in-memory R²:", round(results.loc[results["Model"] == best_name, "R²"].iloc[0], 4))

Reloaded R²: 0.1504
Matches in-memory R²: 0.1504


### Handling Unseen Categories

If a user enters a publisher that wasn't in the training data, `LabelEncoder` raises an error, since it only knows the categories it was fit on. To handle this, we saved a fallback language and publisher in the bundle, and the app substitutes these for unknown values while showing a warning to the user.

## Streamlit Application

The app (`app.py`) reuses the same encoders, median value and feature order saved from training, so predictions stay consistent with what the model actually learned. Users enter raw fields — title, authors, page count, publisher, publication date — and the app applies the same feature engineering steps before calling `predict`.

To run it:

```bash
pip install -r requirements.txt
streamlit run app.py
```

### Why This Matters

If the app used different preprocessing than training, for example a different median or a different feature order, predictions would still come out looking valid, just wrong, with no error or warning. That's worse than a crash, since nothing would flag the problem. This is why we reuse the exact bundle saved from training instead of recomputing anything inside the app.

## Conclusion

This project predicts a book's average rating from metadata only — page count, review activity, publisher, language and a few engineered features — without using the actual text of the books. We dropped rows with a missing target (`average_rating == 0`) and imputed missing `num_pages` values, since those rows still had a usable target.

We built 9 features and compared four regression models on an 80/20 split with `random_state=42`. Gradient Boosting performed best (R² = 0.15), which is a modest but reasonable result, given that book quality, the main driver of ratings, isn't represented in the available metadata.

We saved the trained model together with its encoders and imputation values, and built a Streamlit app that reuses this bundle so predictions stay consistent with training.

### Possible Improvements

- Try target or frequency encoding for `publisher` instead of label encoding.
- Tune Gradient Boosting hyperparameters (`n_estimators`, `learning_rate`, `max_depth`) instead of using defaults.
- Try word count instead of character count for `title_length`.
- Add author-level features, such as the average rating of an author's other books, computed carefully to avoid leaking test data into training.